# Decoder regularisation selected on training-window holdouts

Reuse all saved neural models and preprocessing from the private structured
study. Compare only switch costs 1, 2, 4 and 8 using pooled out-of-fold spellF1.
Save each family's selected cost before reading main validation labels.
All costs, selected outputs and unchanged checkpoint fingerprints are retained.
No neural training or new labels. This is an exploratory follow-up; selected CV
scores are not an independent performance estimate.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys
import torch
assert torch.cuda.is_available()
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/decoder_calibration')
WORK.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0),flush=True)
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
roots=[p.parent for p in INPUT.rglob('DECODER_CALIBRATION.md') if (p.parent/'structured_possession/calibrate.py').is_file()]
assert len(roots)==1
CODE=roots[0]
studies=[p.parent for p in INPUT.rglob('experiment.json')
         if json.loads(p.read_text()).get('kind')=='expanded_labels_typed_events_features_context_ablation']
assert len(studies)==1
SOURCE=studies[0]
source_plan=json.loads((SOURCE/'experiment.json').read_text())
configs=list(INPUT.rglob('structured_study_input.json'))
assert len(configs)==1
LABELS=configs[0].parent
base=[p for p in INPUT.rglob('features.npz') if sha(p)==source_plan['feature_sha256']]
audio=[p for p in INPUT.rglob('audio_cache.npz') if sha(p)==source_plan['audio_sha256']]
assert len(base)==len(audio)==1
FEATURES,AUDIO=base[0],audio[0]
RICH=LABELS/'rich_features.npz'
assert sha(RICH)==source_plan['rich_feature_sha256']
for name in ('train','validation'):
    assert sha(LABELS/f'{name}.csv')==source_plan[f'{name}_sha256']
DEPS=Path('/tmp/barca_decoder_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                '--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                '--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),HF_HUB_OFFLINE='1',
         PYTHONDONTWRITEBYTECODE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')


In [ ]:
RUN=WORK/'calibration'
run('-m','structured_possession.calibrate','--source-study',SOURCE,'--features',FEATURES,
    '--rich-features',RICH,'--audio',AUDIO,'--train-labels',LABELS/'train.csv',
    '--validation-labels',LABELS/'validation.csv','--out',RUN,'--device','cuda','--bootstrap','1000')


In [ ]:
run('-m','structured_possession','predict','--features',RICH,'--audio',AUDIO,
    '--model',RUN/'models/structured','--out',RUN/'structured_regenerated.csv','--device','cuda')
assert (RUN/'structured_regenerated.csv').read_bytes()==(RUN/'structured.csv').read_bytes()
run('-m','temporal_possession','predict','--features',FEATURES,'--audio',AUDIO,
    '--model',RUN/'models/baseline/learned_audio','--out',RUN/'baseline_raw.csv','--device','cuda')
run('tools/apply_match_constraints.py','--features',FEATURES,'--predictions',RUN/'baseline_raw.csv',
    '--out',RUN/'baseline_regenerated.csv')
assert (RUN/'baseline_regenerated.csv').read_bytes()==(RUN/'baseline.csv').read_bytes()
print('Both calibrated model outputs regenerate byte-identically without reference labels.',flush=True)
report=json.loads((RUN/'report.json').read_text())
for family,grid in report['cv'].items():
    for cost,score in grid.items():
        print('OOF',family,'cost',cost,'spellF1',round(score['spell_f1_mean'],4),
              'possessions',score['n_pred_spells'],flush=True)
for name,score in report['scores'].items():
    print('SELECTED',name,'cost',report['selected_costs'][name],
          'spellF1',round(score['spell_f1_mean'],4),'macroF1',round(score['macro_f1'],4),flush=True)
protocol=dict(neural_weights_unchanged=True,selected_before_main_validation=True,
              both_models_regenerated=True,torch=torch.__version__,cuda=torch.version.cuda,
              source_sha256={str(p.relative_to(CODE)):sha(p) for p in (CODE/'structured_possession').glob('*.py')})
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('DECODER CALIBRATION GPU RUN COMPLETE',flush=True)
